# 04_02 Naive Bayes: counting like a probabilist

Naive Bayes classifies by asking: if this were a positive review, how likely would these exact words be? It
answers with a product of word probabilities, counted from the training data. It is the oldest method in this
course and still one of the fastest. By the end you will have worked Bayes' theorem by hand, fixed the book's
word probabilities that added up to more than one, and watched the model call "not bad" 99 percent negative.

**How this notebook works.** Every notebook in this course has the same rhythm:

1. **Recall.** Answer from memory before you look anything up. `ask()` tells you at once whether you were right.
2. **Predict, then run.** Before a cell with a surprise in it, write your prediction into `guess()`. The next cell runs the code and `reveal()` compares.
3. **Worked example, then your turn.** One example is done in full; the next, near-identical one has lines marked `# YOUR CODE HERE`.
4. **Check.** A `check_...()` cell tests what you saved, exactly as the checkpoint will, and says what to fix.

Run cells in order with **Shift+Enter**. If you get lost, **Kernel, Restart Kernel and Run All Cells** starts clean.

Running this in Google Colab? This cell sets it up; in CourseLabs it does nothing.

In [ ]:
# Colab setup. In a CourseLabs session this cell does nothing.
import os, sys
if "google.colab" in sys.modules:
    import importlib, importlib.util, subprocess
    LAB, REPO = "lab-nlp-04-which-classifier-and-why", "/content/nlp-course"
    if not os.path.isdir(REPO):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/fenago/nlp-course.git", REPO], check=True)
    os.chdir(f"{REPO}/{LAB}")
    if not os.path.exists("data"):
        os.symlink("../data", "data")
    os.makedirs("out", exist_ok=True)
    os.environ["NLPLAB_DATA"] = f"{REPO}/data"
    sys.path.insert(0, os.getcwd())
    PIP = {'sentence_transformers': 'sentence-transformers',
           'sklearn': 'scikit-learn',
           'pandas': 'pandas',
           'numpy': 'numpy',
           'matplotlib': 'matplotlib'}
    missing = [spec for mod, spec in PIP.items() if importlib.util.find_spec(mod) is None]
    if missing:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
        importlib.invalidate_caches()
    print(f"Ready: {LAB} and its data are in {os.getcwd()}; installed {len(missing)} package(s).")
elif not os.path.isdir("/opt/nlplab/data") and os.path.isdir("data"):
    # A downloaded copy on your own computer: the helpers read data/ from here.
    os.environ["NLPLAB_DATA"] = os.path.abspath("data")

In [ ]:
import json
import os
import time
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import make_pipeline
import clftools
from nlpcheck import ask, guess, reveal, check_04_02

X_train, X_test, y_train, y_test = clftools.split(clftools.load_sentences())

## 1. Recall

From the last notebook.

**r3.** KNN with K = 1 scored 1.000 on its training sentences. What did that tell you? (a) it is the best K,
(b) nothing: every training sentence is its own nearest neighbour, (c) the data is too easy

**r4.** When does KNN do most of its work? (a) training, (b) both equally, (c) predicting

In [ ]:
ask("r3", "")
ask("r4", "")

## 2. Bayes' theorem, by hand

The book's example. Machine 1 makes 30 bulbs an hour and Machine 2 makes 20. One percent of all bulbs are
defective, and half of the defective ones come from each machine. **What is the probability that a bulb from
Machine 2 is defective?**

Bayes' theorem turns the question round: P(defective | M2) = P(M2 | defective) × P(defective) / P(M2).
Predict first, as a probability.

In [ ]:
guess("defective_given_m2", None)

In [ ]:
p_m2 = 20 / 50                 # the share of all bulbs Machine 2 makes
p_defective = 0.01
p_m2_given_defective = 0.5
answer = p_m2_given_defective * p_defective / p_m2
print(f"P(defective | M2) = {answer:.4f}, or {answer:.2%}")
reveal("defective_given_m2", round(answer, 4))

1.25 percent, higher than the overall 1 percent. Machine 2 makes only 40 percent of the bulbs but half of the
defective ones, so a bulb from Machine 2 is more likely than average to be faulty. Bayes' theorem is the
arithmetic of exactly this kind of reversal: from "how likely is the evidence, given the cause" to "how likely
is the cause, given the evidence".

## 3. The book's eight sentences, and the planted bug

The book trains Naive Bayes by hand on eight sentences, labelled question or statement, then asks about a new
one: *what is the price of the book*. First, count every word in each class.

In [ ]:
rows = [("This is my book", "stmt"), ("They are novels", "stmt"), ("have you read this book", "question"),
        ("who is the author", "question"), ("what are the characters", "question"),
        ("This is how I bought the book", "stmt"), ("I like fictions", "stmt"),
        ("what is your favorite book", "question")]
training = pd.DataFrame(rows, columns=["sent", "class"])

counts = {}
for c in ("stmt", "question"):
    docs = training[training["class"] == c]["sent"]
    vec = CountVectorizer().fit(docs)
    counts[c] = dict(zip(vec.get_feature_names_out(), vec.transform(docs).toarray().sum(axis=0).tolist()))
    print(c, counts[c])

Now turn counts into probabilities: P(word | class). The function below is the book notebook's code. It has one
planted bug that the book printed without noticing. A set of probabilities over every word in a class must add
up to exactly 1. Predict what these add up to.

In [ ]:
def word_probabilities(c):
    # The planted bug, as the book wrote it: divides by the number of DISTINCT words
    return {w: n / len(counts[c]) for w, n in counts[c].items()}

guess("book_probability_sum", None)

In [ ]:
for c in ("stmt", "question"):
    print(c, "sum of probabilities:", round(sum(word_probabilities(c).values()), 3))
reveal("book_probability_sum", round(sum(word_probabilities("stmt").values()), 3))

1.25 and 1.286. Probabilities that add up to more than one are not probabilities. The book divided each count
by the number of **different** words in the class (12 and 14), when it should divide by the **total** number of
words (15 and 18), which is what makes them sum to 1. Three words appear twice in the statements, so the
shortfall is exactly those repeats.

**Fix it**: change the division in `word_probabilities` to divide by the class's total word count,
`sum(counts[c].values())`, and run both cells again. Both sums should print 1.0.

## 4. Smoothing, and the posterior by hand

The new sentence contains *price* and *of*, which never appeared in training. A probability of zero for one word
would make the whole product zero, so Naive Bayes adds one to every count (**Laplace smoothing**):

P(word | class) = (count + 1) / (total words in class + distinct words in the whole training set)

There are 21 distinct words in the whole training set. **Your turn**: finish `smoothed`, then run the cell.

In [ ]:
new = "what is the price of the book".split()
vocab_size = len(CountVectorizer().fit(training["sent"]).vocabulary_)
print("distinct words in training:", vocab_size)

def smoothed(word, c):
    total = sum(counts[c].values())
    return None   # YOUR CODE HERE: (the word's count in class c, or 0, plus 1) / (total + vocab_size)

score = {}
for c in ("stmt", "question"):
    prior = (training["class"] == c).mean()
    probs = [smoothed(w, c) for w in new]
    score[c] = prior * np.prod(probs) if None not in probs else 0.0
    print(c, "prior", prior, "product", score[c])
posterior_question_by_hand = score["question"] / sum(score.values()) if sum(score.values()) else None
print("P(question | sentence) by hand:", posterior_question_by_hand)

When `smoothed` is right, the question class wins, 0.794 against 0.206, which is the book's answer. Now the same
thing with scikit-learn. Predict: will `MultinomialNB` give exactly the same 0.794?

In [ ]:
guess("sklearn_same_as_hand", None)   # "yes" or "no" 

In [ ]:
cv = CountVectorizer()
nb_book = MultinomialNB(alpha=1.0).fit(cv.fit_transform(training["sent"]), training["class"])
probs = dict(zip(nb_book.classes_, nb_book.predict_proba(cv.transform([" ".join(new)]))[0]))
posterior_question_sklearn = float(probs["question"])
print("scikit-learn:", {k: round(v, 3) for k, v in probs.items()})
reveal("sklearn_same_as_hand", "yes" if posterior_question_by_hand and abs(posterior_question_sklearn - posterior_question_by_hand) < 0.001 else "no")

0.819, not 0.794. Same algorithm, same smoothing, different answer, and the reason is the two words nobody had
seen. `CountVectorizer` only has columns for words it met in training, so *price* and *of* simply vanish before
the model sees the sentence. The hand calculation kept them, with the smoothed probability 1 / (total + 21),
which is a little smaller for the question class because it has more words (18 against 15). Each unseen word
therefore nudged the hand answer towards "statement". Neither is wrong; they disagree about what to do with a
word that has no evidence.

Save your numbers:

In [ ]:
os.makedirs("out", exist_ok=True)
sums = {c: sum(word_probabilities(c).values()) for c in ("stmt", "question")}
json.dump({"prob_sum_stmt": sums["stmt"], "prob_sum_question": sums["question"],
           "posterior_question_by_hand": posterior_question_by_hand,
           "posterior_question_sklearn": posterior_question_sklearn},
          open("out/04_02_bayes.json", "w"), indent=1, default=float)
check_04_02()

## 5. On real sentences, and what "naive" costs

The same model on the 2,100 training sentences. It trains in hundredths of a second:

In [ ]:
t = time.time()
nb = make_pipeline(CountVectorizer(), MultinomialNB()).fit(X_train, y_train)
print(f"trained in {time.time() - t:.3f} s, test accuracy {nb.score(X_test, y_test):.3f}")

About 0.81. Now the naive assumption: every word is independent evidence. Predict the probability the model
gives **"not bad"** of being negative.

In [ ]:
guess("not_bad_negative", None)

In [ ]:
for s in ["good", "good good good", "bad", "not good", "not bad", "not bad at all, actually great"]:
    p = nb.predict_proba([s])[0]
    print(f"{s!r:36} negative {p[0]:.3f}   positive {p[1]:.3f}")
reveal("not_bad_negative", round(nb.predict_proba(["not bad"])[0][0], 3))

0.991 negative. To Naive Bayes "not" is a word that appears in negative reviews, and "bad" is another; it
multiplies the two pieces of evidence as if they were unrelated, and becomes very sure. "Good good good" shows
the same mechanism from the other side: the evidence counts three times, and 0.76 becomes 0.97. The
probabilities Naive Bayes prints are often far too confident, even when its answers are right, which is why
you should use its ranking and not trust its numbers.

## 6. Exit ticket

**x3.** Why is Naive Bayes called naive? (a) it uses probability, (b) it uses Bayes' theorem, (c) it assumes
the features are independent

**x5.** Is Naive Bayes (a) supervised or (b) unsupervised?

In [ ]:
ask("x3", "")
ask("x5", "")

Explain it back: why did scikit-learn and the hand calculation disagree about *what is the price of the book*?

*Your explanation:* 